In [27]:
!airflow api-server


Please confirm database migrate (or wait 4 seconds to skip it). Are you sure? [y/N]
[2025-11-26T20:55:41.372+0300] {providers_manager.py:953} INFO - The hook_class 'airflow.providers.standard.hooks.filesystem.FSHook' is not fully initialized (UI widgets will be missing), because the 'flask_appbuilder' package is not installed, however it is not required for Airflow components to work
[2025-11-26T20:55:41.373+0300] {providers_manager.py:953} INFO - The hook_class 'airflow.providers.standard.hooks.package_index.PackageIndexHook' is not fully initialized (UI widgets will be missing), because the 'flask_appbuilder' package is not installed, however it is not required for Airflow components to work
  ____________       _____________
 ____    |__( )_________  __/__  /________      __
____  /| |_  /__  ___/_  /_ __  /_  __ \_ | /| / /
___  ___ |  / _  /   _  __/ _  / / /_/ /_ |/ |/ /
 _/_/  |_/_/  /_/    /_/    /_/  \____/____/|__/
[2025-11-26T20:55:41.392+0300] {api_server_command.py:119} I

In [26]:
import schedule
import time
import datetime
import logging
from pathlib import Path

# Настройка логирования
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('scheduler.log'),
        logging.StreamHandler()
    ]
)

class TaskScheduler:
    def __init__(self):
        self.logger = logging.getLogger(__name__)
        
    def backup_task(self):
        """Задача резервного копирования"""
        try:
            self.logger.info("Запуск задачи резервного копирования")
            # Ваш код для резервного копирования
            time.sleep(2)  # Имитация работы
            self.logger.info("Резервное копирование завершено")
        except Exception as e:
            self.logger.error(f"Ошибка в задаче резервного копирования: {e}")
    
    def cleanup_task(self):
        """Задача очистки временных файлов"""
        try:
            self.logger.info("Запуск задачи очистки")
            # Ваш код для очистки
            time.sleep(1)
            self.logger.info("Очистка завершена")
        except Exception as e:
            self.logger.error(f"Ошибка в задаче очистки: {e}")
    
    def report_task(self):
        """Задача генерации отчетов"""
        try:
            self.logger.info("Генерация ежедневного отчета")
            # Ваш код для генерации отчета
            current_date = datetime.datetime.now().strftime("%Y-%m-%d")
            report_content = f"Отчет за {current_date}\nДанные: ..."
            
            # Сохранение отчета в файл
            report_file = Path(f"report_{current_date}.txt")
            report_file.write_text(report_content, encoding='utf-8')
            
            self.logger.info(f"Отчет сохранен в {report_file}")
        except Exception as e:
            self.logger.error(f"Ошибка при генерации отчета: {e}")

def main():
    scheduler = TaskScheduler()
    
    # Настройка расписания
    schedule.every(3).seconds.do(scheduler.backup_task)
    schedule.every().hour.do(scheduler.cleanup_task)
    schedule.every().day.at("09:00").do(scheduler.report_task)
    schedule.every().day.at("18:00").do(scheduler.report_task)
    
    logging.info("Планировщик задач запущен")
    
    try:
        while True:
            schedule.run_pending()
            time.sleep(1)
    except KeyboardInterrupt:
        logging.info("Планировщик остановлен пользователем")

if __name__ == "__main__":
    main()

[2025-11-26T20:51:37.534+0300] {1471732104.py:66} INFO - Планировщик задач запущен
[2025-11-26T20:51:37.536+0300] {1471732104.py:24} INFO - Запуск задачи резервного копирования
[2025-11-26T20:51:39.541+0300] {1471732104.py:27} INFO - Резервное копирование завершено
[2025-11-26T20:51:39.544+0300] {1471732104.py:24} INFO - Запуск задачи резервного копирования
[2025-11-26T20:51:41.551+0300] {1471732104.py:27} INFO - Резервное копирование завершено
[2025-11-26T20:51:41.554+0300] {1471732104.py:34} INFO - Запуск задачи очистки
[2025-11-26T20:51:42.561+0300] {1471732104.py:37} INFO - Очистка завершена
[2025-11-26T20:51:42.564+0300] {1471732104.py:34} INFO - Запуск задачи очистки
[2025-11-26T20:51:43.570+0300] {1471732104.py:37} INFO - Очистка завершена
[2025-11-26T20:51:44.577+0300] {1471732104.py:24} INFO - Запуск задачи резервного копирования
[2025-11-26T20:51:46.586+0300] {1471732104.py:27} INFO - Резервное копирование завершено
[2025-11-26T20:51:46.589+0300] {1471732104.py:24} INFO - Зап

In [24]:
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime, timedelta
import pandas as pd
import numpy as np

def generate_finance_data():
    """Генерация тестовых финансовых данных"""
    dates = pd.date_range('2024-01-01', periods=100, freq='D')
    data = {
        'date': dates,
        'price': np.random.normal(100, 15, 100).cumsum(),
        'volume': np.random.randint(1000, 10000, 100)
    }
    df = pd.DataFrame(data)
    print(f"Сгенерировано {len(df)} записей")
    return df

def calculate_metrics(**kwargs):
    """Расчет финансовых метрик"""
    ti = kwargs['ti']
    df = ti.xcom_pull(task_ids='generate_data')
    
    df['daily_return'] = df['price'].pct_change()
    df['volatility'] = df['daily_return'].rolling(7).std()
    df['moving_avg'] = df['price'].rolling(7).mean()
    
    metrics = {
        'total_days': len(df),
        'avg_price': df['price'].mean(),
        'max_price': df['price'].max(),
        'min_price': df['price'].min(),
        'avg_volume': df['volume'].mean()
    }
    
    print("Рассчитанные метрики:", metrics)
    return metrics

def detect_anomalies(**kwargs):
    """Обнаружение аномалий в данных"""
    ti = kwargs['ti']
    metrics = ti.xcom_pull(task_ids='calculate_metrics')
    df = ti.xcom_pull(task_ids='generate_data')
    
    # Простая логика обнаружения аномалий
    price_std = df['price'].std()
    price_mean = df['price'].mean()
    
    anomalies = df[abs(df['price'] - price_mean) > 2 * price_std]
    
    print(f"Обнаружено {len(anomalies)} аномалий")
    return len(anomalies)

def generate_report(**kwargs):
    """Генерация финального отчета"""
    ti = kwargs['ti']
    metrics = ti.xcom_pull(task_ids='calculate_metrics')
    anomalies_count = ti.xcom_pull(task_ids='detect_anomalies')
    
    report = {
        'report_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'summary': f"Проанализировано {metrics['total_days']} дней",
        'price_range': f"{metrics['min_price']:.2f} - {metrics['max_price']:.2f}",
        'anomalies_detected': anomalies_count,
        'status': 'COMPLETED'
    }
    
    print("ФИНАЛЬНЫЙ ОТЧЕТ:")
    for key, value in report.items():
        print(f"  {key}: {value}")
    
    return report

# Настройки DAG
default_args = {
    'owner': 'airflow',
    'depends_on_past': False,
    'start_date': datetime(2024, 1, 1),
    'retries': 1,
    'retry_delay': timedelta(minutes=5)
}

# Создание DAG
with DAG(
    'simple_finance_pipeline',
    default_args=default_args,
    description='Простой финансовый пайплайн без SQL',
    schedule=timedelta(minutes=30),
    catchup=False,
    tags=['finance', 'simple']
) as dag:

    # Задача 1: Генерация данных
    generate_task = PythonOperator(
        task_id='generate_data',
        python_callable=generate_finance_data
    )

    # Задача 2: Расчет метрик
    calculate_task = PythonOperator(
        task_id='calculate_metrics',
        python_callable=calculate_metrics
    )

    # Задача 3: Поиск аномалий
    anomalies_task = PythonOperator(
        task_id='detect_anomalies',
        python_callable=detect_anomalies
    )

    # Задача 4: Генерация отчета
    report_task = PythonOperator(
        task_id='generate_report',
        python_callable=generate_report
    )

    # Определение порядка выполнения
    generate_task >> calculate_task >> anomalies_task >> report_task

/var/folders/5d/rj2vhd7s3lgf7m08mvzp24xr0000gn/T/ipykernel_80966/1935294009.py:2 DeprecationWarning: The `airflow.operators.python.PythonOperator` class is deprecated. Please use `'airflow.providers.standard.operators.python.PythonOperator'`.

In [5]:
!airflow db migrate

[2025-11-26T16:47:43.189+0300] {providers_manager.py:953} INFO - The hook_class 'airflow.providers.standard.hooks.filesystem.FSHook' is not fully initialized (UI widgets will be missing), because the 'flask_appbuilder' package is not installed, however it is not required for Airflow components to work
[2025-11-26T16:47:43.191+0300] {providers_manager.py:953} INFO - The hook_class 'airflow.providers.standard.hooks.package_index.PackageIndexHook' is not fully initialized (UI widgets will be missing), because the 'flask_appbuilder' package is not installed, however it is not required for Airflow components to work
DB: sqlite:////Users/ivan7chuk/airflow/airflow.db
Performing upgrade to the metadata database sqlite:////Users/ivan7chuk/airflow/airflow.db
[2025-11-26T16:47:43.239+0300] {migration.py:211} INFO - Context impl SQLiteImpl.
[2025-11-26T16:47:43.240+0300] {migration.py:214} INFO - Will assume non-transactional DDL.
[2025-11-26T16:47:43.240+0300] {migration.py:211} INFO - Context im

In [19]:
import subprocess
import json

# Запуск DAG через командную строку
def run_airflow_dag(dag_id):
    result = subprocess.run([
        'airflow', 'dags', 'trigger', dag_id
    ], capture_output=True, text=True)
    
    if result.returncode == 0:
        print(f"DAG {dag_id} запущен")
        return result.stdout
    else:
        print(f"Ошибка: {result.stderr}")
        return None

# Использование
run_airflow_dag('simple_finance_pipeline')

Ошибка: Traceback (most recent call last):
  File "/Users/ivan7chuk/Library/Python/3.9/bin/airflow", line 8, in <module>
    sys.exit(main())
  File "/Users/ivan7chuk/Library/Python/3.9/lib/python/site-packages/airflow/__main__.py", line 55, in main
    args.func(args)
  File "/Users/ivan7chuk/Library/Python/3.9/lib/python/site-packages/airflow/cli/cli_config.py", line 48, in command
    return func(*args, **kwargs)
  File "/Users/ivan7chuk/Library/Python/3.9/lib/python/site-packages/airflow/utils/cli.py", line 112, in wrapper
    return f(*args, **kwargs)
  File "/Users/ivan7chuk/Library/Python/3.9/lib/python/site-packages/airflow/utils/providers_configuration_loader.py", line 55, in wrapped_function
    return func(*args, **kwargs)
  File "/Users/ivan7chuk/Library/Python/3.9/lib/python/site-packages/airflow/cli/commands/dag_command.py", line 70, in dag_trigger
    message = api_client.trigger_dag(
  File "/Users/ivan7chuk/Library/Python/3.9/lib/python/site-packages/airflow/api/client

In [22]:
!python3 -m airflow dags reserialize

/opt/homebrew/opt/python@3.13/bin/python3.13: No module named airflow.__main__; 'airflow' is a package and cannot be directly executed


In [ ]:
from datetime import datetime, timedelta
from airflow import DAG
from airflow.operators.python import PythonOperator
import pandas as pd
import sqlite3
import requests

def extract_data():
    """Извлечение данных из разных источников"""
    # Чтение из БД
    conn = sqlite3.connect('/tmp/finance_source.db')
    db_data = pd.read_sql("SELECT * FROM stock_prices", conn)
    conn.close()
    
    # Чтение из файла
    file_data = pd.read_csv('/tmp/financial_data.csv')
    
    # Чтение из API
    response = requests.get("https://api.marketdata.com/stocks")
    api_data = pd.DataFrame(response.json()['data'])
    
    return {'db_data': db_data, 'file_data': file_data, 'api_data': api_data}

def transform_data(**kwargs):
    """Трансформация данных"""
    ti = kwargs['ti']
    data = ti.xcom_pull(task_ids='extract_data')
    
    # Объединение и обработка
    merged = pd.concat([
        data['db_data'],
        data['file_data'],
        data['api_data']
    ], ignore_index=True)
    
    # Расчеты
    merged['timestamp'] = datetime.now()
    merged['value_eur'] = merged['value_usd'] * 0.85  # Пример конвертации
    
    # Сохранение в промежуточную БД
    conn = sqlite3.connect('/tmp/processed_data.db')
    merged.to_sql('processed_stocks', conn, if_exists='replace', index=False)
    conn.close()
    
    return len(merged)

def load_results():
    """Загрузка результатов в финальную БД"""
    conn = sqlite3.connect('/tmp/processed_data.db')
    data = pd.read_sql("SELECT * FROM processed_stocks", conn)
    conn.close()
    
    # Финансовые расчеты
    results = data.groupby('ticker').agg({
        'price': ['mean', 'std', 'last'],
        'volume': 'sum',
        'value_eur': 'sum'
    }).round(2)
    
    # Сохранение в финальную БД
    conn = sqlite3.connect('/tmp/final_results.db')
    results.to_sql('financial_results', conn, if_exists='replace')
    conn.close()
    
    return f"Processed {len(results)} tickers"

# Варианты шедулинга:

# 1. Ежедневно в 9 утра по будням
dag1 = DAG(
    'financial_daily',
    description=''
    schedule_interval='0 9 * * 1-5',  # Пн-Пт в 9:00
    start_date=datetime(2024, 1, 1),
    catchup=False
)

# 2. Каждый час в рабочее время
dag2 = DAG(
    'financial_hourly', 
    schedule_interval='0 9-18 * * 1-5',  # Каждый час с 9 до 18 по будням
    start_date=datetime(2024, 1, 1),
    catchup=False
)

# 3. Раз в 15 минут
dag3 = DAG(
    'financial_frequent',
    schedule_interval='*/15 * * * *',  # Каждые 15 минут
    start_date=datetime(2024, 1, 1),
    catchup=False
)

# 4. Раз в неделю
dag4 = DAG(
    'financial_weekly',
    schedule_interval='0 8 * * 1',  # Каждый понедельник в 8:00
    start_date=datetime(2024, 1, 1),
    catchup=False
)

# Создание задач для каждого DAG
def create_tasks(dag):
    extract_task = PythonOperator(
        task_id='extract_data',
        python_callable=extract_data,
        dag=dag
    )
    
    transform_task = PythonOperator(
        task_id='transform',
        python_callable=transform_data,
        dag=dag
    )
    
    load_task = PythonOperator(
        task_id='load',
        python_callable=load_results,
        dag=dag
    )
    
    extract_task >> transform_task >> load_task

# Инициализация всех DAG
create_tasks(dag1)
create_tasks(dag2) 
create_tasks(dag3)
create_tasks(dag4)

In [16]:
import requests
import json
from airflow.api.client.local_client import Client

# Инициализация клиента
client = Client(None, None)

# Создаем DAG прямо в ноутбуке
dag_code = '''
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime, timedelta

def hello_world():
    print("🎉 Hello from Jupyter DAG!")
    return "Success"

with DAG(
    dag_id="jupyter_dag",
    start_date=datetime(2024, 1, 1),
    schedule=timedelta(hours=1),
    catchup=False,
    tags=["jupyter"]
) as dag:

    hello_task = PythonOperator(
        task_id="hello_task",
        python_callable=hello_world
    )
'''

# Сохраняем DAG файл
with open('~/airflow/dags/jupyter_dag.py', 'w') as f:
    f.write(dag_code)

# Запускаем DAG
try:
    result = client.trigger_dag(
        dag_id='jupyter_dag',
        run_id=f'manual_run_{datetime.now().strftime("%Y%m%d_%H%M%S")}',
        conf={}
    )
    print("✅ DAG запущен!")
except Exception as e:
    print(f"❌ Ошибка: {e}")

FileNotFoundError: [Errno 2] No such file or directory: '~/airflow/dags/jupyter_dag.py'